# ParametricStarCoarsener example

This notebook learns one arity-independent sibling family, inspects schema-1
occurrences, and checks exact decoding. Matching labels identify a family;
concrete arity and geometry live in each node's exact `type`.


In [ ]:
from tree_coarsening import ParametricStarCoarsener
from tree_coarsening.utils import make_starburst_dataset

training = make_starburst_dataset(
    n_graphs=3,
    seed=0,
    max_nodes=40,
    n_bursts=2,
    burst_size_range=(4, 7),
)

model = ParametricStarCoarsener(d=3, m=1, model_id="star-example").fit(training)
encoded = model.transform(training[0])
print(f"raw nodes: {training[0].number_of_nodes()}")
print(f"encoded nodes: {encoded.number_of_nodes()}")

In [ ]:
for rule in model.encoder_.rules:
    print(
        "pair=",
        (rule.pattern["parent_label"], rule.pattern["child_label"]),
        "output=",
        rule.output_label,
        "fitting_size=",
        rule.output_fitting_size,
    )

print("vocabulary:", model.encoder_.vocab.as_dict())

In [ ]:
for node, data in encoded.nodes(data=True):
    if isinstance(data["label"], tuple) and data["label"][:1] == ("star",):
        print(
            f"node={node} label={data['label']!r} exact_size={data['size']} "
            f"uids={len(data['super_uids'])} type={data['type']!r}"
        )

for parent, child, data in encoded.edges(data=True):
    assert isinstance(data["attach_map"], tuple)

In [ ]:
def raw_signature(graph):
    nodes = sorted(
        (data["uid"], tuple(sorted(data.items(), key=lambda item: repr(item[0]))))
        for _, data in graph.nodes(data=True)
    )
    edges = sorted(
        (graph.nodes[parent]["uid"], graph.nodes[child]["uid"]) for parent, child in graph.edges
    )
    return nodes, edges


roundtrip = model.decode(encoded)
assert raw_signature(roundtrip) == raw_signature(training[0])
print("roundtrip ok")